In [1]:
import datasets
import numpy as np
import pandas as pd
import os
import sys
import itertools

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# データの前処理

In [3]:
datasets_raw = []
dataset_files = []
for file in os.listdir("/home/yano/rcss/work/SoccerCompetition2025/submission_data"):
    if file.endswith("HELIOS2024_2-vs-YuShan2024_0.tracking.csv"):
        datasets_raw.append(pd.read_csv(f"/home/yano/rcss/work/SoccerCompetition2025/submission_data/{file}"))
        dataset_files.append(file)

for index, dataset in enumerate(datasets_raw):
    datasets_raw[index] = dataset.iloc[-20:]

In [4]:
from pathlib import Path
import pandas as pd

data_dir = Path("/home/yano/rcss/work/SoccerCompetition2025/submission_data")
filename = "HELIOS2024_2-vs-YuShan2024_0.tracking.csv"

dataset_files = [filename]
datasets_raw = [pd.read_csv(data_dir / filename)]

# 読み込んだデータの末尾20行だけを残す
datasets_raw[0] = datasets_raw[0].iloc[-20:].copy()

In [5]:
def one_hot(df: pd.DataFrame) -> pd.DataFrame:
    d = {
        "AEteam": 0,
        "CYRUS": 1,
        "FRA-UNIted": 2,
        "HELIOS2024": 3,
        "ITAndroids": 4,
        "Mars": 5,
        "Oxsy": 6,
        "R2D2": 7,
        "RoboCIn": 8,
        "YuShan2024": 9,
    }
    df["l_name"] = df["l_name"].map(d)
    df["r_name"] = df["r_name"].map(d)
    return df


def drop(df: pd.DataFrame) -> pd.DataFrame:
    agents = [f"l{id}" for id in range(1, 12)] + [f"r{id}" for id in range(1, 12)]
    b = ["vx", "vy", "t", "body", "neck", "vwidth", "stamina"]
    drop_cols = [f"{a}_{b}" for a, b in itertools.product(agents, b)]
    drop_cols += ["b_vx", "b_vy"]
    drop_cols += [
        "#",
        "cycle",
        "stopped",
        "playmode",
        "l_score",
        "l_pen_score",
        "r_score",
        "r_pen_score",
    ]
    return df.drop(columns=drop_cols)


def frame(df: pd.DataFrame) -> pd.DataFrame:
    df["frame"] = df.reset_index().index
    return df


def process_data(df: pd.DataFrame) -> list[pd.DataFrame]:
    df = one_hot(df)
    df = frame(df)
    return df


for index, dataset in enumerate(datasets_raw):
    datasets_raw[index] = process_data(dataset)

In [6]:
class MinMax:
    def __init__(self, min, max):
        self.min = min
        self.max = max
        if self.min >= self.max:
            raise ValueError("min must be less than max")

    def __call__(self, x):
        return (x - self.min) / (self.max - self.min)

    def inverse(self, x):
        return x * (self.max - self.min) + self.min

    def __repr__(self):
        return f"MinMax({self.min}, {self.max})"


def swap_rl(df):
    df["l_name"], df["r_name"] = df["r_name"], df["l_name"]
    df["b_x"] *= -1

    for i in range(1, 12):
        l_x, r_x = f"l{i}_x", f"r{i}_x"
        l_y, r_y = f"l{i}_y", f"r{i}_y"

        df[l_x], df[r_x] = -df[r_x].values, -df[l_x].values
        df[l_y], df[r_y] = df[r_y].values, df[l_y].values

    return df


min_max_d = np.load("/mnt/data1/yano/work/SoccerCompetition2025/datas_20260828_015504/min_max_d.npy", allow_pickle=True).item()

In [7]:
def clean_and_merge_datasets(datas):
    data_list = []
    for data in datas:
        df = pd.DataFrame(data)
        data_list.append(df)
    datas = pd.concat(data_list)
    return datas


def name_onehot(dfs):
    for i in range(10):
        dfs[f"l_name_{i}"] = dfs["l_name"] == i
        dfs[f"r_name_{i}"] = dfs["r_name"] == i
        dfs[f"l_name_{i}"] = (dfs["l_name"] == i).astype(int)
        dfs[f"r_name_{i}"] = (dfs["r_name"] == i).astype(int)
    return dfs


def drop_unnecessary_columns(dfs):
    dfs = dfs.drop(
        columns=[
            "#",
            "cycle",
            "stopped",
            "playmode",
            "l_name",
            "r_name",
            # "goal_type",
            "l_score",
            "r_score",
            "l_pen_score",
            "r_pen_score",
        ]
    )
    return dfs


def min_max_normalize(dfs, min_max_d=None):
    if min_max_d is None:
        min_max_d = {col: MinMax(min(dfs[col]), max(dfs[col])) for col in dfs.columns}
    for col in dfs.columns:
        dfs[col] = min_max_d[col](dfs[col])
    return dfs, min_max_d


def revert_min_max_normalize(dfs, min_max_d):
    for col, scaler in min_max_d.items():
        dfs[col] = scaler.inverse(dfs[col])
    return dfs


def divide_dataframe(dfs, df_size=50):
    df_list = []

    for i in range(0, len(dfs), df_size):
        df = dfs.iloc[i : i + df_size]
        df_list.append(df)
    return df_list


def list_to_numpy(dfs: list) -> np.ndarray:
    cols = dfs[0].columns
    return np.array([df.values for df in dfs]).astype(np.float32), cols


def revert_numpy_from_list(dfs: np.ndarray, cols) -> list[pd.DataFrame]:
    return [pd.DataFrame(df, columns=cols) for df in dfs]


submission = datasets_raw
submission = clean_and_merge_datasets(submission)
display(submission.iloc[:100])
submission = name_onehot(submission)
submission = drop_unnecessary_columns(submission)
submission, _ = min_max_normalize(submission, min_max_d)
submission = divide_dataframe(submission, df_size=20)

submission, cols = list_to_numpy(submission)

,#,cycle,stopped,playmode,l_name,l_score,l_pen_score,r_name,r_score,r_pen_score,b_x,b_y,b_vx,b_vy,l1_t,l1_x,l1_y,l1_vx,l1_vy,l1_body,l1_neck,l1_vwidth,l1_stamina,l2_t,l2_x,l2_y,l2_vx,l2_vy,l2_body,l2_neck,l2_vwidth,l2_stamina,l3_t,l3_x,l3_y,l3_vx,l3_vy,l3_body,l3_neck,l3_vwidth,l3_stamina,l4_t,l4_x,l4_y,l4_vx,l4_vy,l4_body,l4_neck,l4_vwidth,l4_stamina,l5_t,l5_x,l5_y,l5_vx,l5_vy,l5_body,l5_neck,l5_vwidth,l5_stamina,l6_t,l6_x,l6_y,l6_vx,l6_vy,l6_body,l6_neck,l6_vwidth,l6_stamina,l7_t,l7_x,l7_y,l7_vx,l7_vy,l7_body,l7_neck,l7_vwidth,l7_stamina,l8_t,l8_x,l8_y,l8_vx,l8_vy,l8_body,l8_neck,l8_vwidth,l8_stamina,l9_t,l9_x,l9_y,l9_vx,l9_vy,l9_body,l9_neck,l9_vwidth,l9_stamina,l10_t,l10_x,l10_y,l10_vx,l10_vy,l10_body,l10_neck,l10_vwidth,l10_stamina,l11_t,l11_x,l11_y,l11_vx,l11_vy,l11_body,l11_neck,l11_vwidth,l11_stamina,r1_t,r1_x,r1_y,r1_vx,r1_vy,r1_body,r1_neck,r1_vwidth,r1_stamina,r2_t,r2_x,r2_y,r2_vx,r2_vy,r2_body,r2_neck,r2_vwidth,r2_stamina,r3_t,r3_x,r3_y,r3_vx,r3_vy,r3_body,r3_neck,r3_vwidth,r3_stamina,r4_t,r4_x,r4_y,r4_vx,r4_vy,r4_body,r4_neck,r4_vwidth,r4_stamina,r5_t,r5_x,r5_y,r5_vx,r5_vy,r5_body,r5_neck,r5_vwidth,r5_stamina,r6_t,r6_x,r6_y,r6_vx,r6_vy,r6_body,r6_neck,r6_vwidth,r6_stamina,r7_t,r7_x,r7_y,r7_vx,r7_vy,r7_body,r7_neck,r7_vwidth,r7_stamina,r8_t,r8_x,r8_y,r8_vx,r8_vy,r8_body,r8_neck,r8_vwidth,r8_stamina,r9_t,r9_x,r9_y,r9_vx,r9_vy,r9_body,r9_neck,r9_vwidth,r9_stamina,r10_t,r10_x,r10_y,r10_vx,r10_vy,r10_body,r10_neck,r10_vwidth,r10_stamina,r11_t,r11_x,r11_y,r11_vx,r11_vy,r11_body,r11_neck,r11_vwidth,r11_stamina,frame
3976,3977,3738,0,play_on,3,0,0,9,0,0,30.0464,-15.1062,0.6569,-0.1178,0,-34.7747,-2.4871,0.0000,-0.0000,-22.411,25,180,8000.0,1,-0.7567,-25.3625,0.1774,-0.0414,-11.397,90,60,7632.43,2,1.1564,21.3983,-0.0121,-0.2653,-99.728,48,60,7617.37,8,-5.5669,14.0930,0.0791,0.0086,4.629,-31,60,6339.32,14,-7.3906,-15.2007,0.0549,-0.0278,-23.0,40,60,8000.00,10,-4.1036,-0.4923,0.1194,-0.0399,-16.797,13,60,7962.4,9,10.3596,-15.0402,0.3125,-0.0274,-4.697,90,120,6657.64,13,11.6369,6.3616,0.3510,-0.0701,-14.321,90,120,6027.73,4,19.5927,18.6029,0.1907,-0.2377,-52.068,-46,120,6510.94,15,28.8995,-14.8192,0.3304,-0.0490,-9.311,90,120,6710.02,12,17.5770,-8.5422,0.4153,-0.0746,-6.775,-7,60,6766.45,0,49.6851,-4.3618,0.0168,-0.2311,-90.918,-10,120,7945,10,31.4859,-1.1888,0.3831,-0.1834,-25.827,-83,60,5747.11,12,27.1712,-13.0285,0.2554,0.0559,12.757,-39,60,6212.91,1,27.8972,9.0009,0.4390,-0.1791,-18.666,-90,120,6516.38,2,30.0796,-17.1313,0.3259,0.1797,9.482,25,60,6804.04,13,21.0245,-5.0487,0.3785,-0.1223,-23.058,-45,60,4500.53,14,13.6018,6.0044,0.2485,-0.1420,-29.575,90,120,4801.54,9,13.9192,-14.7548,0.2974,-0.1056,-20.197,-90,60,4408.25,4,3.3035,22.3500,0.2307,-0.0665,-15.103,40,180,6135.65,15,3.9527,-19.9232,0.1198,0.0111,-55.520,-90,120,7185.97,8,-1.4870,-0.0268,0.0849,-0.0629,-37.086,-40,120,6579.05,0
3977,3978,3739,0,play_on,3,0,0,9,0,0,30.6780,-15.2147,0.5937,-0.1020,0,-34.7747,-2.4871,0.0000,-0.0000,-22.411,25,180,8000.0,1,-0.3378,-25.4562,0.1972,-0.0441,-11.397,90,60,7632.43,2,1.1304,21.0022,-0.0122,-0.1857,-99.728,48,60,7641.67,8,-5.3645,14.0972,0.0790,0.0016,4.629,-64,60,6360.16,14,-7.2226,-15.2962,0.0544,-0.0309,-23.0,22,60,8000.00,10,-3.8628,-0.5720,0.0987,-0.0327,-16.797,14,60,7983.2,9,11.2509,-15.1199,0.3185,-0.0285,-4.697,55,120,6599.17,13,12.5339,6.1464,0.3922,-0.0941,-14.321,-15,120,5974.91,4,20.1795,17.8226,0.2052,-0.2729,-52.068,-46,120,6451.66,15,29.9057,-15.0269,0.3910,-0.0807,-9.311,90,60,6651.97,12,18.4702,-8.5647,0.3695,-0.0093,6.686,80,60,6722.61,0,49.7033,-4.5770,0.0073,-0.0861,-90.918,-10,120,7990,10,32.3853,-1.6714,0.3689,-0.1979,-25.827,-64,60,5688.71,12,27.9883,-12.8404,0.3381,0.0778,12.757,89,60,6156.98,1,28.8322,8.6533,0.4401,-0.1636,-18.666,-60,120,6458.28,2,30.6185,-16.6433,0.2527,0.2288,9.482,90,60,6752.64,13,21.8417,-5.4288,0.3573,-0.1662,-23.058,-90,60,4447.71,14,13.8391,5.8703,0.0768,-0.0434,-13.253,-88,120,4843.82,9,14.1984,-14.8636,0.0998,-0.0389,-2.952,-18,60,4449.78,4,4.0401,22.1483,0.2576,-0.0705

# モデルの読み込み

In [20]:
import joblib
import os

import visualizer

from tqdm.notebook import tqdm
import datasets
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch import nn
import torchmetrics
import pytorch_lightning as pl

PROJECT_NAME = "RobocupTrajectoryPrediction_0902_1642"
MODEL_NAME = "BiGRU"
GROUP_NAME = "BiGRU_1"
HIDDEN_DIM = 256
NUM_LAYER = 4
LEARNING_RATE = 0.001

torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
import joblib
import os

import visualizer

from tqdm.notebook import tqdm
import datasets
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch import nn
import torchmetrics
import pytorch_lightning as pl

PROJECT_NAME = "RobocupTrajectoryPrediction_DataExpanded"
MODEL_NAME = "BiGRU_dataExpanded_2"
GROUP_NAME = "BiGRU"
HIDDEN_DIM = 256
NUM_LAYER = 4
LEARNING_RATE = 0.001

torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [21]:
class EuclideanDistance(torchmetrics.Metric):
    def __init__(self, cols, min_max_d, **kwargs):
        super().__init__(**kwargs)
        self.cols = cols
        self.min_max_d = min_max_d
        self.add_state("sum", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("count", default=torch.tensor(0), dist_reduce_fx="sum")
        self._mins = torch.tensor(
            [min_max_d[col].min for col in cols], dtype=torch.float32
        )
        self._maxs = torch.tensor(
            [min_max_d[col].max for col in cols], dtype=torch.float32
        )
        self._scale = self._maxs - self._mins

    def update(self, preds, target):
        indices = torch.cat([torch.tensor([0]), torch.arange(5, 201, 9)])
        final_preds = preds[:, -1, :]
        final_target = target[:, -1, :]

        inversed_preds = torch.zeros_like(final_preds)
        inversed_target = torch.zeros_like(final_target)

        mins = self._mins.to(device=device, dtype=final_preds.dtype)
        scale = self._scale.to(device=device, dtype=final_preds.dtype)

        inversed_preds = final_preds.clone()
        inversed_target = final_target.clone()
        n_cols = len(self.cols)
        inversed_preds[:, :n_cols] = final_preds[:, :n_cols] * scale + mins
        inversed_target[:, :n_cols] = final_target[:, :n_cols] * scale + mins

        errors = torch.sqrt(
            (inversed_preds[:, indices] - inversed_target[:, indices]) ** 2
            + (inversed_preds[:, indices + 1] - inversed_target[:, indices + 1]) ** 2
        )
        self.sum += torch.sum(errors)
        self.count += errors.numel()

    def compute(self):
        return self.sum / self.count

In [22]:
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger


class LitBiGRU(pl.LightningModule):
    def __init__(
        self, input_dim, hidden_dim, output_dim, num_layers, seq_length, lr=0.001
    ):
        super().__init__()
        self.save_hyperparameters()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim * seq_length)
        self.criterion = nn.MSELoss()
        self.euclidean_distance = EuclideanDistance(cols, min_max_d)
        self.train_losses = []
        self.val_losses = []

    def forward(self, x):
        out, _ = self.gru(x)
        last_out = out[:, -1, :]
        output = self.fc(last_out)
        return output.view(-1, 30, self.hparams.output_dim)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.log(
            "train_euclidean_distance",
            self.euclidean_distance(y_hat, y),
            on_step=False,
            on_epoch=True,
        )
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True)
        self.log(
            "val_euclidean_distance",
            self.euclidean_distance(y_hat, y),
            on_step=False,
            on_epoch=True,
        )
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

In [23]:
# load model
model_path = "/home/yano/rcss/work/SoccerCompetition2025/models/BiGRU_1/BiGRU_1_20260922161617.pth"

model = LitBiGRU(
    input_dim=submission.shape[2],
    hidden_dim=HIDDEN_DIM,
    output_dim=submission.shape[2],
    num_layers=NUM_LAYER,
    seq_length=30,
    lr=LEARNING_RATE,
)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

# model = LitBiGRU.load_from_checkpoint(model_path)

LitBiGRU(
  (gru): GRU(223, 256, num_layers=4, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=512, out_features=6690, bias=True)
  (criterion): MSELoss()
  (euclidean_distance): EuclideanDistance()
)

In [25]:
# show model summary
print(model)

LitBiGRU(
  (gru): GRU(223, 256, num_layers=4, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=512, out_features=6690, bias=True)
  (criterion): MSELoss()
  (euclidean_distance): EuclideanDistance()
)


# 提出データの予測

In [18]:
import visualizer


def f():
    target = submission
    with torch.no_grad():
        model.eval()
        preds = model(torch.tensor(target).to(device=device))
        preds = preds.cpu().numpy()

    tareget_df = []
    for i in range(len(preds)):
        df = pd.DataFrame(preds[i], columns=cols)
        for j in df.columns:
            df[j] = min_max_d[j].inverse(df[j])
        tareget_df.append(df)

    # concat
    # display(tareget_df)

    dfs = []
    for index, file_name in enumerate(dataset_files):
        pred_df = tareget_df[index]
        sub_df = pd.read_csv(f"/mnt/data1/yano/work/SoccerCompetition2025/submission_data/HELIOS2024_2-vs-YuShan2024_0.tracking.csv")

        sub_cols = sub_df.columns
        df = pd.concat([sub_df, pred_df])
        df = df[sub_cols]
        dfs.append(df)

    # index rest
    for index, df in enumerate(dfs):
        df.index = range(1, len(df) + 1)

    # df[#] = index
    for index, df in enumerate(dfs):
        df["#"] = df.index

    for index, df in enumerate(dfs):
        dfs[index] = df[
            [
                "#",
                "l1_x",
                "l1_y",
                "l2_x",
                "l2_y",
                "l3_x",
                "l3_y",
                "l4_x",
                "l4_y",
                "l5_x",
                "l5_y",
                "l6_x",
                "l6_y",
                "l7_x",
                "l7_y",
                "l8_x",
                "l8_y",
                "l9_x",
                "l9_y",
                "l10_x",
                "l10_y",
                "l11_x",
                "l11_y",
                "r1_x",
                "r1_y",
                "r2_x",
                "r2_y",
                "r3_x",
                "r3_y",
                "r4_x",
                "r4_y",
                "r5_x",
                "r5_y",
                "r6_x",
                "r6_y",
                "r7_x",
                "r7_y",
                "r8_x",
                "r8_y",
                "r9_x",
                "r9_y",
                "r10_x",
                "r10_y",
                "r11_x",
                "r11_y",
                "b_x",
                "b_y",
            ]
        ]
    # last 30
    for index, df in enumerate(dfs):
        dfs[index] = df.iloc[-30:]
    # save csv
    os.makedirs("/mnt/data1/yano/work/SoccerCompetition2025/submission_data_out", exist_ok=True)
    for index, df in enumerate(dfs):
        df.to_csv(
            f"/mnt/data1/yano/work/SoccerCompetition2025/submission_data_out/{dataset_files[index]}", index=False
        )

    return dfs


dfs = f()

In [14]:
from pathlib import Path
import os
import pandas as pd
import torch

INPUT_DIR = Path("/mnt/data1/yano/work/SoccerCompetition2025/submission_data")
OUTPUT_DIR = Path("/mnt/data1/yano/work/SoccerCompetition2025/submission_data_out")
TARGET_FILE = "HELIOS2024_2-vs-YuShan2024_0.tracking.csv"


def f():
    # submission は対象ファイル1件分の入力データである前提
    with torch.no_grad():
        model.eval()
        preds = model(
            torch.tensor(submission, dtype=torch.float32).to(device)
        ).cpu().numpy()

    # バッチ先頭（対象ファイル1件）の予測結果をDataFrameにする
    pred_df = pd.DataFrame(preds[0], columns=cols)

    # 正規化を元の値域へ戻す
    for col in pred_df.columns:
        pred_df[col] = min_max_d[col].inverse(pred_df[col])

    # 元データを読み込み、予測値を後ろに連結する
    sub_df = pd.read_csv(INPUT_DIR / TARGET_FILE)
    df = pd.concat([sub_df, pred_df], ignore_index=True)

    # 提出に必要な列だけ残す
    output_columns = [
        "#",
        "l1_x", "l1_y", "l2_x", "l2_y", "l3_x", "l3_y",
        "l4_x", "l4_y", "l5_x", "l5_y", "l6_x", "l6_y",
        "l7_x", "l7_y", "l8_x", "l8_y", "l9_x", "l9_y",
        "l10_x", "l10_y", "l11_x", "l11_y",
        "r1_x", "r1_y", "r2_x", "r2_y", "r3_x", "r3_y",
        "r4_x", "r4_y", "r5_x", "r5_y", "r6_x", "r6_y",
        "r7_x", "r7_y", "r8_x", "r8_y", "r9_x", "r9_y",
        "r10_x", "r10_y", "r11_x", "r11_y",
        "b_x", "b_y",
    ]
    df = df[output_columns]

    # 提出対象の予測30フレームだけ残し、# を1から振り直す
    df = df.iloc[-30:].copy()
    df["#"] = range(1, len(df) + 1)

    # 保存
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / TARGET_FILE
    df.to_csv(output_path, index=False)

    return df


df = f()